# Teste do Modelo 1: Amazon Chronos

In [1]:
import pandas as pd
import torch
import os

# --- 1. Configurações ---
DATA_DIR = "../../data"
HORIZONTE_PREVISAO = 14

# --- 2. Carregar Dados ---
print("Carregando dados...")
hist_path = os.path.join(DATA_DIR, "hist.parquet")
df_hist = pd.read_parquet(hist_path)
print("Dados históricos carregados:")
print(df_hist.tail())

# --- 3. Definir dispositivo ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsando dispositivo: {device}")

Carregando dados...
Dados históricos carregados:
                target  day_of_week  month
date                                      
2023-04-06   91.026088            3      4
2023-04-07   82.740630            4      4
2023-04-08   87.401396            5      4
2023-04-09  100.150267            6      4
2023-04-10  112.014606            0      4

Usando dispositivo: cpu


In [2]:
# ==============================================================================
# TESTE 1: Amazon Chronos
# ==============================================================================
# Covariáveis: Não. Chronos é um modelo "zero-shot".
# ==============================================================================

try:
    from chronos import ChronosPipeline

    print("\n--- Testando Amazon Chronos ---")

    # 1. Carregar o pipeline (modelo)
    pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-small",
        device_map=device,
        torch_dtype=torch.bfloat16,
    )

    # 2. Preparar dados
    # Chronos precisa apenas da série temporal (target) como um tensor
    context_tensor = torch.tensor(df_hist['target'].values)

    # 3. Rodar a previsão
    print(f"Rodando previsão para {HORIZONTE_PREVISAO} passos...")
    forecast = pipeline.predict(
        context_tensor,
        HORIZONTE_PREVISAO,
        limit_prediction_length=False # Garante que ele preveja exatamente H passos
    )

    # forecast[0] contém o tensor previsto
    print("Previsão (primeiros 5 valores):", forecast[0].numpy().round(2)[:5])
    print("Teste do Chronos concluído.\n")

except ImportError:
    print("Chronos não instalado. Pulando teste.")
except Exception as e:
    print(f"Erro ao rodar Chronos: {e}")


--- Testando Amazon Chronos ---


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Rodando previsão para 14 passos...


Previsão (primeiros 5 valores): [[115.37 109.3   97.15  86.11  98.26 107.64 119.79 120.34 113.16 102.12
   92.19 104.33 115.92 126.41]
 [113.71 107.09  93.84  83.35  89.43 103.23 113.71 119.23 111.51  95.5
   86.67  92.74 106.54 119.79]
 [110.95 102.12  93.84  83.35  89.43 102.67 116.47 111.51 101.57  94.95
   85.01  89.43 103.78 117.03]
 [114.27 108.19  93.84  86.67  93.84 105.99 113.71 117.58 110.4   95.5
   89.98  98.26 112.06 117.03]
 [117.58 110.4   99.91  88.32  94.39 108.75 119.23 123.1  116.47 105.43
   93.29 101.02 114.27 124.2 ]]
Teste do Chronos concluído.

